<table align="center">
  <td align="center"><a target="_blank" href="http://introtodeeplearning.com">
        <img src="https://i.ibb.co/Jr88sn2/mit.png" style="padding-bottom:5px;" />
      Visit MIT Deep Learning</a></td>
  <td align="center"><a target="_blank" href="https://colab.research.google.com/github/MITDeepLearning/introtodeeplearning/blob/master/lab2/PT_Part1_MNIST.ipynb">
        <img src="https://i.ibb.co/2P3SLwK/colab.png"  style="padding-bottom:5px;" />Run in Google Colab</a></td>
  <td align="center"><a target="_blank" href="https://github.com/MITDeepLearning/introtodeeplearning/blob/master/lab2/PT_Part1_MNIST.ipynb">
        <img src="https://i.ibb.co/xfJbPmL/github.png"  height="70px" style="padding-bottom:5px;"  />View Source on GitHub</a></td>
</table>

# Copyright Information

In [ ]:
# Copyright 2026 MIT Introduction to Deep Learning. All Rights Reserved.
#
# Licensed under the MIT License. You may not use this file except in compliance
# with the License. Use and/or modification of this code outside of MIT Introduction
# to Deep Learning must reference:
#
# © MIT Introduction to Deep Learning
# http://introtodeeplearning.com
#

# Bài thực hành 2: Thị giác máy tính

# Phần 1: Phân loại chữ số viết tay MNIST

Trong phần đầu tiên của bài thực hành này, chúng ta sẽ xây dựng và huấn luyện một mạng thần kinh tích chập (CNN) để phân loại các chữ số viết tay từ tập dữ liệu nổi tiếng [MNIST](http://yann.lecun.com/exdb/mnist/). Tập dữ liệu MNIST bao gồm 60.000 hình ảnh huấn luyện và 10.000 hình ảnh kiểm tra. Các lớp phân loại của chúng ta là các chữ số từ 0 đến 9.

Trước tiên, hãy tải xuống kho lưu trữ của khóa học, cài đặt các thư viện phụ thuộc và nhập các gói thư viện liên quan mà chúng ta sẽ cần cho bài thực hành này.

In [ ]:
# Import PyTorch and other relevant libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchsummary import summary

# MIT introduction to deep learning package
!pip install mitdeeplearning --quiet
import mitdeeplearning as mdl

# other packages
import matplotlib.pyplot as plt
import numpy as np
import random
from tqdm import tqdm

Chúng ta cũng sẽ cài đặt Comet. Nếu bạn đã làm theo hướng dẫn từ Bài thực hành 1, bạn sẽ có tài khoản Comet được thiết lập sẵn. Nhập API key của bạn vào bên dưới.

In [ ]:
!pip install comet_ml > /dev/null 2>&1
import comet_ml
# TODO: ENTER YOUR API KEY HERE!!
COMET_API_KEY = ""

# Check that we are using a GPU, if not switch runtimes
#   using Runtime > Change Runtime Type > GPU
assert torch.cuda.is_available(), "Please enable GPU from runtime settings"
assert COMET_API_KEY != "", "Please insert your Comet API Key"

# Set GPU for computation
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# start a first comet experiment for the first part of the lab
comet_ml.init(project_name="6S191_lab2_part1_NN")
comet_model_1 = comet_ml.Experiment()

## 1.1 Tập dữ liệu MNIST

Hãy tải xuống, tải dữ liệu và hiển thị một vài mẫu ngẫu nhiên từ đó:

In [ ]:
# Download and transform the MNIST dataset
transform = transforms.Compose([
    # Convert images to PyTorch tensors which also scales data from [0,255] to [0,1]
    transforms.ToTensor()
])

# Download training and test datasets
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

Đối tượng dữ liệu MNIST trong PyTorch không đơn thuần là một tensor hay một mảng. Nó là một tập dữ liệu có thể lặp (iterable dataset) giúp tải từng mẫu (cặp hình ảnh - nhãn) tại một thời điểm hoặc theo từng lô (batch). Trong phần sau của bài thực hành này, chúng ta sẽ định nghĩa một DataLoader tiện lợi để xử lý dữ liệu theo từng lô.

In [ ]:
image, label = train_dataset[0]
print(image.size())  # For a tensor: torch.Size([1, 28, 28])
print(label)  # For a label: integer (e.g., 5)

Tập dữ liệu huấn luyện của chúng ta bao gồm các hình ảnh thang độ xám kích thước 28x28 của các chữ số viết tay.

Hãy trực quan hóa xem một số hình ảnh này và các nhãn huấn luyện tương ứng của chúng trông như thế nào.

In [ ]:
plt.figure(figsize=(10,10))
random_inds = np.random.choice(60000,36)
for i in range(36):
    plt.subplot(6, 6, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    image_ind = random_inds[i]
    image, label = train_dataset[image_ind]
    plt.imshow(image.squeeze(), cmap=plt.cm.binary)
    plt.xlabel(label)
comet_model_1.log_figure(figure=plt)

## 1.2 Mạng thần kinh cho việc phân loại chữ số viết tay

Trước tiên, chúng ta sẽ xây dựng một mạng thần kinh đơn giản gồm hai lớp liên kết đầy đủ (fully connected) và áp dụng nó vào nhiệm vụ phân loại chữ số. Mạng của chúng ta cuối cùng sẽ xuất ra một phân phối xác suất trên 10 lớp chữ số (0-9). Kiến trúc đầu tiên mà chúng ta sẽ xây dựng được mô tả bên dưới:

![alt_text](https://raw.githubusercontent.com/MITDeepLearning/introtodeeplearning/master/lab2/img/mnist_2layers_arch.png "Kiến trúc CNN cho phân loại MNIST")

### Kiến trúc mạng thần kinh liên kết đầy đủ
Để định nghĩa kiến trúc của mạng thần kinh liên kết đầy đủ đầu tiên này, chúng ta sẽ một lần nữa sử dụng các mô-đun `torch.nn`, định nghĩa mô hình bằng [`nn.Sequential`](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html). Hãy lưu ý cách đầu tiên chúng ta sử dụng một lớp [`nn.Flatten`](https://pytorch.org/docs/stable/generated/torch.nn.Flatten.html), giúp làm phẳng đầu vào để có thể đưa vào mô hình.

Trong khối mã tiếp theo, bạn sẽ định nghĩa các lớp liên kết đầy đủ của mạng đơn giản này.

In [ ]:
def build_fc_model():
    fc_model = nn.Sequential(
        # First define a Flatten layer
        nn.Flatten(),

        # '''TODO: Define the activation function for the first fully connected (Dense/Linear) layer.'''
        nn.Linear(28 * 28, 128),
        '''TODO'''

        '''TODO: Define the second Linear layer to output the classification probabilities'''
        )
    return fc_model

fc_model_sequential = build_fc_model()

Khi chúng ta tiếp tục, bạn có thể muốn thực hiện các thay đổi đối với kiến trúc đã được định nghĩa ở trên. **Lưu ý rằng để cập nhật mô hình sau này, bạn cần chạy lại cell phía trên để khởi tạo lại mô hình.**

Hãy lùi lại một bước và suy nghĩ về mạng chúng ta vừa tạo. Lớp đầu tiên trong mạng này, `nn.Flatten`, chuyển đổi định dạng của hình ảnh từ mảng 2 chiều (28 x 28 pixel) thành mảng 1 chiều có kích thước 28 * 28 = 784 pixel. Bạn có thể coi lớp này như việc dỡ các hàng pixel trong ảnh và xếp thẳng hàng chúng lại. Không có tham số học tập nào trong lớp này; nó chỉ định dạng lại dữ liệu.

Sau khi các pixel được làm phẳng, mạng bao gồm một chuỗi gồm hai lớp `nn.Linear`. Đây là các lớp liên kết đầy đủ. Lớp `nn.Linear` đầu tiên có 128 nút (hoặc neuron). Lớp thứ hai (và là lớp cuối cùng) (do bạn định nghĩa!) sẽ trả về một mảng điểm xác suất có tổng bằng 1. Mỗi nút chứa một điểm số cho biết xác suất hình ảnh hiện tại thuộc về một trong các lớp chữ số viết tay.

Đó là định nghĩa cho mô hình liên kết đầy đủ của chúng ta!

### Sử dụng phương pháp kế thừa (subclassing) trong PyTorch

Hãy nhớ lại trong Bài thực hành 1, chúng ta đã khám phá cách tạo ra các mô hình linh hoạt hơn bằng cách kế thừa lớp [`nn.Module`](https://pytorch.org/docs/stable/generated/torch.nn.Module.html). Kỹ thuật định nghĩa mô hình này thường được sử dụng phổ biến hơn trong PyTorch. Chúng ta sẽ thực hành sử dụng cách tiếp cận kế thừa này để định nghĩa các mô hình của mình trong phần còn lại của bài thực hành.

In [ ]:
# Define the fully connected model
class FullyConnectedModel(nn.Module):
    def __init__(self):
        super(FullyConnectedModel, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28 * 28, 128)

        # '''TODO: Define the activation function for the first fully connected layer'''
        self.relu = # TODO

        # '''TODO: Define the second Linear layer to output the classification probabilities'''
        self.fc2 = # TODO

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)

        # '''TODO: Implement the rest of forward pass of the model using the layers you have defined above'''
        '''TODO'''

        return x

fc_model = FullyConnectedModel().to(device) # send the model to GPU

### Các chỉ số đánh giá và tham số huấn luyện của mô hình

Trước khi huấn luyện mô hình, chúng ta cần định nghĩa các thành phần quản lý hiệu suất và hướng dẫn quá trình học tập của nó. Chúng bao gồm hàm mất mát (loss function), bộ tối ưu hóa (optimizer) và các chỉ số đánh giá (metrics):

* *Hàm mất mát (Loss function)* — Định nghĩa cách chúng ta đo lường mức độ chính xác của mô hình trong quá trình huấn luyện. Như đã đề cập trong bài giảng, khi huấn luyện chúng ta muốn giảm thiểu hàm này để "lái" mô hình đi đúng hướng.
* *Bộ tối ưu hóa (Optimizer)* — Định nghĩa cách mô hình được cập nhật dựa trên dữ liệu nó nhìn thấy và hàm mất mát của nó.
* *Chỉ số đánh giá (Metrics)* — Ở đây chúng ta có thể định nghĩa các chỉ số muốn sử dụng để giám sát các bước huấn luyện và kiểm tra. Trong ví dụ này, chúng ta sẽ định nghĩa và xem xét *độ chính xác (accuracy)*, tỷ lệ hình ảnh được phân loại chính xác.

Chúng ta sẽ bắt đầu bằng cách sử dụng bộ tối ưu hóa giảm độ dốc ngẫu nhiên (SGD) được khởi tạo với tốc độ học (learning rate) là 0.1. Vì đang thực hiện tác vụ phân loại danh mục, chúng ta sẽ muốn sử dụng [cross entropy loss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html).

Bạn sẽ muốn thử nghiệm với cả việc lựa chọn bộ tối ưu hóa và tốc độ học, đồng thời đánh giá xem những lựa chọn này ảnh hưởng thế nào đến độ chính xác của mô hình sau khi huấn luyện.

In [ ]:
'''TODO: Experiment with different optimizers and learning rates. How do these affect
    the accuracy of the trained model? Which optimizers and/or learning rates yield
    the best performance?'''
# Define loss function and optimizer
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(fc_model.parameters(), lr=0.1)

### Huấn luyện mô hình

Bây giờ chúng ta đã sẵn sàng huấn luyện mô hình, việc này bao gồm nạp dữ liệu huấn luyện (`train_dataset`) vào mô hình và yêu cầu nó học các liên kết giữa hình ảnh và nhãn. Chúng ta cũng cần định nghĩa kích thước lô (batch size) và số kỷ nguyên (epochs), hoặc số lần lặp lại trên toàn bộ tập dữ liệu MNIST để sử dụng trong quá trình huấn luyện. Tập dữ liệu này bao gồm các bộ dữ liệu dạng bộ (hình ảnh, nhãn) mà chúng ta sẽ truy cập lặp đi lặp lại theo từng lô.

Trong Bài thực hành 1, chúng ta đã thấy cách sử dụng phương thức [`.backward()`](https://pytorch.org/docs/stable/generated/torch.Tensor.backward.html) để tối ưu hóa tổn thất và huấn luyện mô hình với giảm độ dốc ngẫu nhiên. Trong phần này, chúng ta sẽ định nghĩa một hàm để huấn luyện mô hình sử dụng `.backward()` và `optimizer.step()` để tự động cập nhật các tham số mô hình (trọng số và độ lệch) như chúng ta đã thấy trong Bài thực hành 1.

Hãy nhớ lại, chúng tôi đã đề cập trong Phần 1.1 rằng tập dữ liệu MNIST có thể được truy cập lặp đi lặp lại theo từng lô. Ở đây, chúng ta sẽ định nghĩa một [`DataLoader`](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader) của PyTorch để hỗ trợ thực hiện việc đó.

In [ ]:
# Create DataLoaders for batch processing
BATCH_SIZE = 64
trainset_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
testset_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
def train(model, dataloader, criterion, optimizer, epochs):
    model.train()  # Set the model to training mode
    for epoch in range(epochs):
        total_loss = 0
        correct_pred = 0
        total_pred = 0

        for images, labels in trainset_loader:
            # Move tensors to GPU so compatible with model
            images, labels = images.to(device), labels.to(device)

            # Forward pass
            outputs = fc_model(images)

            # Clear gradients before performing backward pass
            optimizer.zero_grad()
            # Calculate loss based on model predictions
            loss = loss_function(outputs, labels)
            # Backpropagate and update model parameters
            loss.backward()
            optimizer.step()

            # multiply loss by total nos. of samples in batch
            total_loss += loss.item()*images.size(0)

            # Calculate accuracy
            predicted = torch.argmax(outputs, dim=1)  # Get predicted class
            correct_pred += (predicted == labels).sum().item()  # Count correct predictions
            total_pred += labels.size(0) # Count total predictions

        # Compute metrics
        total_epoch_loss = total_loss / total_pred
        epoch_accuracy = correct_pred / total_pred
        print(f"Epoch {epoch + 1}, Loss: {total_epoch_loss}, Accuracy: {epoch_accuracy:.4f}")


In [ ]:
# TODO: Train the model by calling the function appropriately
EPOCHS = 5
train('''TODO''') # TODO

comet_model_1.end()

Khi mô hình huấn luyện, các chỉ số mất mát và độ chính xác sẽ được hiển thị. Với 5 kỷ nguyên (epochs) và tốc độ học 0.01, mô hình liên kết đầy đủ này sẽ đạt được độ chính xác khoảng 0.97 (hoặc 97%) trên dữ liệu huấn luyện.

### Đánh giá độ chính xác trên tập dữ liệu kiểm tra

Bây giờ sau khi đã huấn luyện mô hình, chúng ta có thể yêu cầu nó đưa ra dự đoán trên một tập kiểm tra mà nó chưa từng thấy trước đây. Trong ví dụ này, việc lặp qua `testset_loader` cho phép chúng ta truy cập các hình ảnh kiểm tra và nhãn kiểm tra của mình. Và để đánh giá độ chính xác, chúng ta có thể kiểm tra xem các dự đoán của mô hình có khớp với các nhãn từ bộ nạp này hay không.

Vì hiện tại chúng ta đã huấn luyện mô hình, chúng ta sẽ sử dụng trạng thái đánh giá (eval state) của mô hình trên tập dữ liệu kiểm tra.

In [ ]:
'''TODO: Use the model we have defined in its eval state to complete
and call the evaluate function, and calculate the accuracy of the model'''

def evaluate(model, dataloader, loss_function):
    # Evaluate model performance on the test dataset
    model.eval()
    test_loss = 0
    correct_pred = 0
    total_pred = 0
    # Disable gradient calculations when in inference mode
    with torch.no_grad():
        for images, labels in testset_loader:
            # TODO: ensure evalaution happens on the GPU
            images, labels = # TODO

            # TODO: feed the images into the model and obtain the predictions (forward pass)
            outputs = # TODO

            loss = loss_function(outputs, labels)

            # TODO: Calculate test loss
            test_loss += # TODO

           '''TODO: make a prediction and determine whether it is correct!'''
            # TODO: identify the digit with the highest probability prediction for the images in the test dataset.
            predicted = # torch.argmax('''TODO''')

            # TODO: tally the number of correct predictions
            correct_pred += TODO

            # TODO: tally the total number of predictions
            total_pred += TODO

    # Compute average loss and accuracy
    test_loss /= total_pred
    test_acc = correct_pred / total_pred
    return test_loss, test_acc

# TODO: call the evaluate function to evaluate the trained model!!
test_loss, test_acc = # TODO

print('Test accuracy:', test_acc)

Bạn có thể quan sát thấy độ chính xác trên tập dữ liệu kiểm tra thấp hơn một chút so với độ chính xác trên tập dữ liệu huấn luyện. Sự chênh lệch giữa độ chính xác huấn luyện và độ chính xác kiểm tra này là một ví dụ về hiện tượng *quá khớp (overfitting)*, khi một mô hình học máy hoạt động kém hơn trên dữ liệu mới so với dữ liệu huấn luyện của nó.

Độ chính xác cao nhất bạn có thể đạt được với mô hình liên kết đầy đủ đầu tiên này là bao nhiêu? Vì tác vụ phân loại chữ số viết tay khá đơn giản, bạn có thể tự hỏi làm thế nào chúng ta có thể làm tốt hơn...

![Deeper...](https://i.kym-cdn.com/photos/images/newsfeed/000/534/153/f87.jpg)

## 1.3 Mạng thần kinh tích chập (CNN) cho việc phân loại chữ số viết tay

Như chúng ta đã thấy trong bài giảng, mạng thần kinh tích chập (CNN) đặc biệt phù hợp cho nhiều tác vụ khác nhau trong thị giác máy tính và đã đạt được độ chính xác gần như hoàn hảo trên tập dữ liệu MNIST. Bây giờ chúng ta sẽ xây dựng một mạng CNN gồm hai lớp tích chập và các lớp gom nhóm (pooling), theo sau là hai lớp liên kết đầy đủ, và cuối cùng xuất ra phân phối xác suất trên 10 lớp chữ số (0-9). Mạng CNN chúng ta sẽ xây dựng được mô tả bên dưới:

![alt_text](https://raw.githubusercontent.com/MITDeepLearning/introtodeeplearning/master/lab2/img/convnet_fig.png "Kiến trúc CNN cho phân loại MNIST")

### Định nghĩa mô hình CNN

Chúng ta sẽ sử dụng cùng tập dữ liệu huấn luyện và kiểm tra như trước, đồng thời tiến hành tương tự như mạng liên kết đầy đủ để định nghĩa và huấn luyện mô hình CNN mới. Để làm điều này, chúng ta sẽ khám phá hai lớp mà chúng ta chưa từng gặp trước đây: bạn có thể sử dụng [`nn.Conv2d`](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) để định nghĩa các lớp tích chập và [`nn.MaxPool2d`](https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html) để định nghĩa các lớp gom nhóm. Sử dụng các tham số được hiển thị trong kiến trúc mạng ở trên để định nghĩa các lớp này và xây dựng mô hình CNN. Bạn có thể chọn sử dụng `nn.Sequential` hoặc kế thừa `nn.Module` tùy theo sở thích.

In [ ]:
### Basic CNN in PyTorch ###

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        # TODO: Define the first convolutional layer
        self.conv1 = # TODO

        # TODO: Define the first max pooling layer
        self.pool1 = # TODO

        # TODO: Define the second convolutional layer
        self.conv2 = # TODO

        # TODO: Define the second max pooling layer
        self.pool2 = # TODO

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(36 * 5 * 5, 128)
        self.relu = nn.ReLU()

        # TODO: Define the Linear layer that outputs the classification
        # logits over class labels. Remember that CrossEntropyLoss operates over logits.
        self.fc2 = # TODO


    def forward(self, x):
        # First convolutional and pooling layers
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool1(x)

        # '''TODO: Implement the rest of forward pass of the model using the layers you have defined above'''
        #     '''hint: this will involve another set of convolutional/pooling layers and then the linear layers'''
        '''TODO'''

        return x

# Instantiate the model
cnn_model = CNN().to(device)
# Initialize the model by passing some data through
image, label = train_dataset[0]
image = image.to(device).unsqueeze(0)  # Add batch dimension → Shape: (1, 1, 28, 28)
output = cnn_model(image)
# Print the model summary
print(cnn_model)

### Huấn luyện và kiểm tra mô hình CNN

Trước đó trong bài thực hành, chúng ta đã định nghĩa hàm `train`. Thân hàm rất hữu ích vì nó cho phép chúng ta kiểm soát mô hình huấn luyện và ghi lại các hoạt động tính toán vi phân trong quá trình huấn luyện bằng cách tính toán các độ dốc qua `loss.backward()`. Bạn có thể nhớ lại đã thấy điều này trong Bài thực hành 1 Phần 1.

Chúng ta sẽ sử dụng cùng một khung này để huấn luyện `cnn_model` của mình bằng cách sử dụng giảm độ dốc ngẫu nhiên. Bạn có thể tự do triển khai các phần sau có hoặc không có các hàm train và evaluate mà chúng ta đã định nghĩa ở trên. Điều quan trọng nhất là hiểu cách thao tác trên thân các hàm đó để huấn luyện và kiểm tra mô hình.

Như chúng ta đã làm ở trên, chúng ta có thể định nghĩa hàm mất mát, bộ tối ưu hóa và tính toán độ chính xác của mô hình. Định nghĩa một bộ tối ưu hóa và tốc độ học tùy chọn. Hãy tự do sửa đổi nếu bạn thấy phù hợp để tối ưu hóa hiệu suất mô hình của mình.

In [ ]:
# Rebuild the CNN model
cnn_model = CNN().to(device)

# Define hyperparams
batch_size = 64
epochs = 7
optimizer = optim.SGD(cnn_model.parameters(), lr=1e-2)

# TODO: instantiate the cross entropy loss function
loss_function = # TODO

# Redefine trainloader with new batch size parameter (tweak as see fit if optimizing)
trainset_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
testset_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
loss_history = mdl.util.LossHistory(smoothing_factor=0.95) # to record the evolution of the loss
plotter = mdl.util.PeriodicPlotter(sec=2, xlabel='Iterations', ylabel='Loss', scale='semilogy')

# Initialize new comet experiment
comet_ml.init(project_name="6.s191lab2_part1_CNN")
comet_model_2 = comet_ml.Experiment()

if hasattr(tqdm, '_instances'): tqdm._instances.clear() # clear if it exists

# Training loop!
cnn_model.train()

for epoch in range(epochs):
    total_loss = 0
    correct_pred = 0
    total_pred = 0

    # First grab a batch of training data which our data loader returns as a tensor
    for idx, (images, labels) in enumerate(tqdm(trainset_loader)):
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        # TODO: feed the images into the model and obtain the predictions
        logits = # TODO

        # TODO: compute the categorical cross entropy loss using the predicted logits
        loss = # TODO

        # Get the loss and log it to comet and the loss_history record
        loss_value = loss.item()
        comet_model_2.log_metric("loss", loss_value, step=idx)
        loss_history.append(loss_value) # append the loss to the loss_history record
        plotter.plot(loss_history.get())

        # Backpropagation/backward pass
        '''TODO: Compute gradients for all model parameters and propagate backwads
            to update model parameters. remember to reset your optimizer!'''
        # TODO: reset optimizer
        # TODO: compute gradients
        # TODO: update model parameters

        # Get the prediction and tally metrics
        predicted = torch.argmax(logits, dim=1)
        correct_pred += (predicted == labels).sum().item()
        total_pred += labels.size(0)

    # Compute metrics
    total_epoch_loss = total_loss / total_pred
    epoch_accuracy = correct_pred / total_pred
    print(f"Epoch {epoch + 1}, Loss: {total_epoch_loss}, Accuracy: {epoch_accuracy:.4f}")

comet_model_2.log_figure(figure=plt)

### Đánh giá mô hình CNN

Bây giờ chúng ta đã huấn luyện mô hình, hãy đánh giá nó trên tập dữ liệu kiểm tra.

In [ ]:
'''TODO: Evaluate the CNN model!'''
test_loss, test_acc = evaluate('''TODO''')

print('Test accuracy:', test_acc)

Độ chính xác cao nhất bạn có thể đạt được khi sử dụng mô hình CNN là bao nhiêu, và độ chính xác của mô hình CNN so với độ chính xác của mạng liên kết đầy đủ đơn giản như thế nào? Bộ tối ưu hóa và tốc độ học nào có vẻ tối ưu nhất để huấn luyện mô hình CNN?

Hãy thoải mái nhấp vào các liên kết Comet để điều tra các đường cong huấn luyện/độ chính xác cho mô hình của bạn.

### Đưa ra dự đoán với mô hình CNN

Với mô hình đã được huấn luyện, chúng ta có thể sử dụng nó để đưa ra dự đoán về một số hình ảnh.

In [ ]:
test_image, test_label = test_dataset[0]
test_image = test_image.to(device).unsqueeze(0)

# put the model in evaluation (inference) mode
cnn_model.eval()
predictions_test_image = cnn_model(test_image)

Với lệnh gọi hàm này, mô hình đã dự đoán nhãn của hình ảnh đầu tiên trong tập kiểm tra. Hãy cùng xem qua dự đoán:

In [ ]:
print(predictions_test_image)

Như bạn có thể thấy, một dự đoán là một mảng gồm 10 số. Hãy nhớ rằng đầu ra của mô hình là một phân phối trên 10 lớp chữ số. Do đó, những con số này mô tả khả năng dự đoán của mô hình rằng hình ảnh tương ứng với mỗi chữ số trong số 10 chữ số khác nhau.

Hãy xem xét chữ số có khả năng xảy ra cao nhất cho hình ảnh đầu tiên trong tập dữ liệu kiểm tra:

In [ ]:
'''TODO: identify the digit with the highest likelihood prediction for the first
    image in the test dataset. '''
predictions_value = predictions_test_image.cpu().detach().numpy() #.cpu() to copy tensor to memory first
prediction = # TODO
print(prediction)

Vì vậy, mô hình tự tin nhất rằng hình ảnh này là một "???". Chúng ta có thể kiểm tra nhãn kiểm tra (hãy nhớ, đây là danh tính thực sự của chữ số) để xem dự đoán này có chính xác không:

In [ ]:
print("Label of this digit is:", test_label)
plt.imshow(test_image[0,0,:,:].cpu(), cmap=plt.cm.binary)
comet_model_2.log_figure(figure=plt)

Đúng vậy! Hãy trực quan hóa kết quả phân loại trên tập dữ liệu MNIST. Chúng ta sẽ vẽ các hình ảnh từ tập dữ liệu kiểm tra cùng với nhãn dự đoán của chúng, cũng như một biểu đồ biểu diễn xác suất dự đoán cho mỗi chữ số.

Hãy nhớ rằng trong PyTorch, tập dữ liệu MNIST thường được truy cập bằng cách sử dụng DataLoader để lặp qua tập kiểm tra theo các lô nhỏ hơn, dễ quản lý hơn. Bằng cách nối các dự đoán, nhãn kiểm tra và hình ảnh kiểm tra từ mỗi lô, trước tiên chúng ta sẽ tích lũy dần dần tất cả dữ liệu cần thiết cho việc trực quan hóa vào các biến riêng lẻ để quan sát các dự đoán của mô hình.

In [ ]:
# Initialize variables to store all data
all_predictions = []
all_labels = []
all_images = []

# Process test set in batches
with torch.no_grad():
    for images, labels in testset_loader:
        outputs = cnn_model(images)

        # Apply softmax to get probabilities from the predicted logits
        probabilities = torch.nn.functional.softmax(outputs, dim=1)

        # Get predicted classes
        predicted = torch.argmax(probabilities, dim=1)

        all_predictions.append(probabilities)
        all_labels.append(labels)
        all_images.append(images)

all_predictions = torch.cat(all_predictions)  # Shape: (total_samples, num_classes)
all_labels = torch.cat(all_labels)            # Shape: (total_samples,)
all_images = torch.cat(all_images)            # Shape: (total_samples, 1, 28, 28)

# Convert tensors to NumPy for compatibility with plotting functions
predictions = all_predictions.cpu().numpy()  # Shape: (total_samples, num_classes)
test_labels = all_labels.cpu().numpy()       # Shape: (total_samples,)
test_images = all_images.cpu().numpy()       # Shape: (total_samples, 1, 28, 28)

In [ ]:
#@title Change the slider to look at the model's predictions! { run: "auto" }

image_index = 79 #@param {type:"slider", min:0, max:100, step:1}
plt.subplot(1,2,1)
mdl.lab2.plot_image_prediction(image_index, predictions, test_labels, test_images)
plt.subplot(1,2,2)
mdl.lab2.plot_value_prediction(image_index, predictions, test_labels)
comet_model_2.log_figure(figure=plt)

Chúng ta cũng có thể vẽ một số hình ảnh cùng với các dự đoán của chúng, trong đó các nhãn dự đoán chính xác có màu xanh lam và các nhãn dự đoán không chính xác có màu xám. Con số này đưa ra tỷ lệ phần trăm độ tự tin (trên 100) cho nhãn dự đoán. Lưu ý rằng mô hình có thể rất tự tin vào một dự đoán sai!

In [ ]:
# Plots the first X test images, their predicted label, and the true label
# Color correct predictions in blue, incorrect predictions in red
num_rows = 5
num_cols = 4
num_images = num_rows*num_cols
plt.figure(figsize=(2*2*num_cols, 2*num_rows))
for i in range(num_images):
  plt.subplot(num_rows, 2*num_cols, 2*i+1)
  mdl.lab2.plot_image_prediction(i, predictions, test_labels, test_images)
  plt.subplot(num_rows, 2*num_cols, 2*i+2)
  mdl.lab2.plot_value_prediction(i, predictions, test_labels)
comet_model_2.log_figure(figure=plt)
comet_model_2.end()

## 1.5 Kết luận
Trong phần này của bài thực hành, bạn đã có cơ hội trải nghiệm các bộ phân loại MNIST khác nhau với các kiến trúc khác nhau (chỉ gồm các lớp liên kết đầy đủ, CNN) và thử nghiệm xem các siêu tham số khác nhau ảnh hưởng thế nào đến độ chính xác (tốc độ học, v.v.). Phần tiếp theo của bài thực hành sẽ khám phá một ứng dụng khác của CNN, nhận diện khuôn mặt, và một số hạn chế của hệ thống AI trong thế giới thực, chẳng hạn như vấn đề về sự thiên vị.